# Private LoRA16 continuation from verified step 665
Attach original private Task1 data and pa3-lora16-step665.zip as a private dataset. Stop prior jobs before saved background launch. This resumes 150 additional updates with a 1800s bound; 585 total updates remain. Prior required checkpoints312/625 remain on the original Modal volume; the local restore contains665 only. Preserve actual saved output ZIP before another continuation.


In [ ]:
# Set only after the previous Kaggle draft is stopped and no training is active.
BATCH_HANDOFF_CONFIRMED = False
assert BATCH_HANDOFF_CONFIRMED, 'Stop the prior draft/session, then confirm handoff before Save & Run All'


In [ ]:
import subprocess, sys, pathlib, json, zipfile, urllib.request
REPO_COMMIT = "cf7f042114a0a924403ed55ef9ee5982946ecf38"
work = pathlib.Path("/kaggle/working")
source = work / "pa3-source"
if not source.exists():
    archive = work / "pa3-source.zip"
    urllib.request.urlretrieve(f"https://github.com/sohauv/atml-pa3/archive/{REPO_COMMIT}.zip", archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall(work)
    (work / f"atml-pa3-{REPO_COMMIT}").rename(source)
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0", "transformers==4.51.3", "accelerate==1.6.0", "datasets==3.5.0", "numpy==2.2.5"])
print("Pinned source prepared:", REPO_COMMIT)


In [ ]:
# Verify all original input bytes; tolerate only byte-identical duplicate copies.
import sys, pathlib, zipfile, hashlib
sys.path.insert(0, str(source))
from scripts.kaggle_task1 import unpack_inputs, select_original_input
EXPECTED_TRAIN_SHA = '644a46f9ab7a8d368499b2054229cc7324614d97596fbdee1ac3ab5cbb584de9'
archives = list(pathlib.Path('/kaggle/input').rglob('pa3-task1-input.zip'))
if len(archives) == 1:
    input_archive = archives[0]
else:
    original = select_original_input('/kaggle/input', EXPECTED_TRAIN_SHA)
    input_archive = work / 'pa3-task1-input.zip'
    if not input_archive.exists():
        with zipfile.ZipFile(input_archive, 'x', zipfile.ZIP_DEFLATED) as z:
            for name in ('train.jsonl', 'metadata.json', 'source_ids.json'):
                z.write(original / name, 'task1/' + name)
data_dir = unpack_inputs(input_archive, work / 'pa3/data')
assert hashlib.file_digest((data_dir / 'train.jsonl').open('rb'), 'sha256').hexdigest() == EXPECTED_TRAIN_SHA
print('Original training data checksum verified:', data_dir, flush=True)


In [ ]:
import pathlib, json, shutil, hashlib
EXPECTED = {'lora16-20261030/resume-0665-1793318754217356917/complete.json': '9191a218878981031df31ffddf53c172282758b0991a483d827920a19c5ecf6a', 'lora16-20261030/resume-0665-1793318754217356917/projected_factors.pt': 'a26818cd4ba760cc66ef1d7e424140d9a80f9609df0e1022beb5240abab8c26c', 'lora16-20261030/resume-0665-1793318754217356917/recovery.pt': '4e4d934c29611430dadbe3683a2b35edf7799e51c1b602ff9973f6cbae7c531d', 'lora16-20261030/resume-0665-1793318754217356917/summary.json': 'd6537b8d14c7779ffccb8295a9bf9d113d02be7e4935bebce2802ce24a0b87eb'}
roots = [p.parent.parent for p in pathlib.Path('/kaggle/input').rglob('complete.json') if p.parent.name == 'resume-0665-1793318754217356917']
assert len(roots) == 1, 'Attach the private pa3-lora16-step665 input exactly once'
root = roots[0]
run = work / 'pa3/runs/lora16-20261030'
assert not run.exists(), 'Fresh background job required; refusing overwrite'
for name, digest in EXPECTED.items():
    file = root / str(pathlib.PurePosixPath(name).relative_to('lora16-20261030'))
    assert file.is_file() and hashlib.file_digest(file.open('rb'), 'sha256').hexdigest() == digest, str(file)
shutil.copytree(root, run)
checkpoint = run / 'resume-0665-1793318754217356917'
summary = json.loads((checkpoint / 'summary.json').read_text())
assert summary['update'] == 665 and summary['successful_optimizer_updates'] == 665 and summary['skipped_updates'] == 0
print('VERIFIED_RESTORE', summary, flush=True)


In [ ]:
import subprocess, sys, json, zipfile, hashlib
command = [sys.executable, '-m', 'scripts.kaggle_task1', '--archive', str(input_archive), '--method', 'lora16', '--run-id', 'lora16-20261030', '--seconds', '1800', '--max-updates', '150', '--resume', str(checkpoint), '--handoff-verified']
print('START_BOUNDED_RECOVERY_FROM', checkpoint, flush=True)
try:
    subprocess.check_call(command, cwd=source)
finally:
    status = json.loads((run / 'status.json').read_text())
    saved = run / pathlib.Path(status['checkpoint']).name
    assert (saved / 'complete.json').is_file()
    result = json.loads((saved / 'summary.json').read_text())
    assert 665 <= result['update'] <= 815
    archive = work / ('lora16-backup-step-%04d.zip' % result['update'])
    with zipfile.ZipFile(archive, 'x', zipfile.ZIP_DEFLATED) as backup:
        for file in run.rglob('*'):
            if file.is_file():
                backup.write(file, str(pathlib.Path('lora16-recovery') / file.relative_to(run)))
    with zipfile.ZipFile(archive) as backup:
        assert backup.testzip() is None
    evidence = {'status': status, 'summary': result, 'archive': str(archive), 'archive_bytes': archive.stat().st_size, 'archive_sha256': hashlib.file_digest(archive.open('rb'), 'sha256').hexdigest()}
    (work / 'checkpoint-backup-evidence.json').write_text(json.dumps(evidence, indent=2))
    print('DOWNLOAD_AND_VERIFY_BEFORE_NEXT_RUN', json.dumps(evidence), flush=True)
